<a href="https://colab.research.google.com/github/Khalidsyfullah/USplitVQA/blob/main/New_BioMedClip_Federated_Learning_VizWiz.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import re
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "open_clip_torch", "transformers", "datasets", "openpyxl", "tqdm"])
import os, random, time, copy; import numpy as np; from collections import Counter; from tqdm.auto import tqdm; import warnings; warnings.filterwarnings('ignore')
import torch, torch.nn as nn; from torch.utils.data import Dataset, DataLoader

device=torch.device('cuda' if torch.cuda.is_available() else 'cpu'); print(f"Device: {device}")
if device.type=='cuda': print(f"GPU: {torch.cuda.get_device_name(0)}")
SEED=42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
OUTPUT_DIR="/content/results"; os.makedirs(OUTPUT_DIR,exist_ok=True)

D=768; HEADS=8; FL=2; DROP=0.15; NC=5; ROUNDS=15; LEP=3; BS=32; FLR=1e-4; WD=1e-4; LS=0.1; MAV=300; MAF=3

# ─────────────────────────────────────────────────────────────────────
# 4. VizWiz  (e.g., Eldon/VizWiz or any HF mirror)
# ─────────────────────────────────────────────────────────────────────
# ~31,000 QA pairs from blind users · real-world photos
# Highly noisy: unanswerable questions, poor image quality,
# answers from 10 crowd annotators. Very diverse open-ended answers
# (objects, colors, text reading, brands, counts, etc.)

def normalize_answer_vizwiz(ans: str) -> str:
    """Normalize VizWiz answers."""
    ans = ans.strip().lower()
    ans = re.sub(r'[^\w\s\-/.,]', '', ans)
    ans = re.sub(r'\s+', ' ', ans).strip()

    # ── Unanswerable / unsuitable canonicalization ──
    unanswerable_set = {
        'unanswerable', 'unsuitable', 'unsuitable image',
        'not answerable', 'cant answer', 'cannot answer',
        'i dont know', 'i don\'t know', 'i do not know',
        'unable to answer', 'not sure', 'unclear',
        'unreadable', 'cannot be determined', 'cant be determined',
        'can not be determined', 'not clear', 'blurry',
        'too blurry', 'too dark', 'no answer', 'na', 'n/a',
        'cannot tell', 'cant tell', 'hard to tell',
        'impossible to tell', 'nothing', 'not possible',
    }
    if ans in unanswerable_set:
        return 'unanswerable'

    # ── Yes / No ──
    yes_set = {'yes', 'yes.', 'yeah', 'yep', 'y', 'correct', 'true',
               'yes it is', 'yes, it is', 'yea', 'ya'}
    no_set  = {'no', 'no.', 'nope', 'n', 'false', 'incorrect', 'negative',
               'no it is not', 'nah'}
    if ans in yes_set:
        return 'yes'
    if ans in no_set:
        return 'no'

    # ── Numeric canonicalization ──
    word_to_num = {'zero': '0', 'one': '1', 'two': '2', 'three': '3',
                   'four': '4', 'five': '5', 'six': '6', 'seven': '7',
                   'eight': '8', 'nine': '9', 'ten': '10',
                   'eleven': '11', 'twelve': '12', 'thirteen': '13',
                   'fourteen': '14', 'fifteen': '15', 'twenty': '20',
                   'thirty': '30', 'forty': '40', 'fifty': '50',
                   'hundred': '100'}
    if ans in word_to_num:
        return word_to_num[ans]

    # ── Color normalization (very common in VizWiz) ──
    color_map = {
        'blue': 'blue', 'light blue': 'blue', 'dark blue': 'blue',
        'navy': 'blue', 'navy blue': 'blue', 'royal blue': 'blue',
        'red': 'red', 'dark red': 'red', 'light red': 'red',
        'maroon': 'red', 'crimson': 'red', 'burgundy': 'red',
        'green': 'green', 'light green': 'green', 'dark green': 'green',
        'lime': 'green', 'olive': 'green',
        'yellow': 'yellow', 'light yellow': 'yellow', 'gold': 'yellow',
        'golden': 'yellow',
        'orange': 'orange',
        'pink': 'pink', 'light pink': 'pink', 'hot pink': 'pink',
        'magenta': 'pink',
        'purple': 'purple', 'violet': 'purple', 'lavender': 'purple',
        'brown': 'brown', 'tan': 'brown', 'beige': 'brown',
        'khaki': 'brown',
        'white': 'white', 'off white': 'white', 'cream': 'white',
        'ivory': 'white',
        'black': 'black', 'dark': 'black',
        'gray': 'gray', 'grey': 'gray', 'silver': 'gray',
        'light gray': 'gray', 'light grey': 'gray',
        'dark gray': 'gray', 'dark grey': 'gray',
    }
    if ans in color_map:
        return color_map[ans]

    # ── Common object synonyms ──
    object_map = {
        'cellphone': 'phone', 'cell phone': 'phone', 'mobile': 'phone',
        'mobile phone': 'phone', 'smartphone': 'phone', 'iphone': 'phone',
        'tv': 'television', 'television': 'television',
        'laptop': 'laptop', 'computer': 'laptop', 'notebook': 'laptop',
        'can': 'can', 'cans': 'can', 'tin': 'can',
        'bottle': 'bottle', 'bottles': 'bottle',
        'box': 'box', 'boxes': 'box', 'package': 'box',
        'shirt': 'shirt', 'tshirt': 'shirt', 't-shirt': 'shirt',
        't shirt': 'shirt', 'tee shirt': 'shirt',
        'pants': 'pants', 'trousers': 'pants', 'jeans': 'pants',
        'shoe': 'shoe', 'shoes': 'shoe', 'sneaker': 'shoe',
        'sneakers': 'shoe',
        'remote': 'remote', 'remote control': 'remote',
        'glasses': 'glasses', 'eyeglasses': 'glasses',
        'sunglasses': 'sunglasses',
        'soda': 'soda', 'pop': 'soda', 'soft drink': 'soda',
        'coke': 'coca cola', 'coca-cola': 'coca cola',
        'pepsi': 'pepsi', 'dr pepper': 'dr pepper',
        'cat': 'cat', 'cats': 'cat', 'kitten': 'cat',
        'dog': 'dog', 'dogs': 'dog', 'puppy': 'dog',
        'dollar': 'dollar', 'dollars': 'dollar',
        'cent': 'cent', 'cents': 'cent',
    }
    if ans in object_map:
        return object_map[ans]

    # ── Remove articles and fillers ──
    ans = re.sub(r'^(the|a|an|its|it is|this is|that is|it\'s|i think)\s+', '', ans)
    ans = re.sub(r'\s+', ' ', ans).strip()

    # ── Remove trailing period ──
    ans = ans.rstrip('.')

    return ans


# ── VizWiz ──
print("\n"+"="*60+"\nLOADING VizWiz\n"+"="*60)
from datasets import load_dataset; ds=load_dataset('lmms-lab/VizWiz-VQA')
def maj(ans):
    if not ans: return ''
    c=[str(a.get('answer','') if isinstance(a,dict) else a).strip().lower() for a in ans]
    c=[a for a in c if a and a not in ('unanswerable','unsuitable')]
    return Counter(c).most_common(1)[0][0] if c else ''
samples=[]
for s in tqdm(ds['val'],desc="VizWiz"):
    try:
        img=s.get('image'); q=str(s.get('question',''))
        if not img or not q: continue
        a=s.get('answer'); a=str(a).strip().lower() if a is not None else maj(s.get('answers',[]))
        a = normalize_answer_vizwiz(a)
        if not a or a in ('unanswerable','unsuitable',''): continue
        samples.append({'image':img.convert('RGB'),'question':q,'answer':a})
    except: continue
del ds; random.shuffle(samples); sp=int(len(samples)*0.8); train_s,test_s=samples[:sp],samples[sp:]
aa=[s['answer'] for s in samples]; cc=Counter(aa); filt=[(a,c) for a,c in cc.most_common() if c>=MAF][:MAV]
av={'<unk>':0}
for i,(a,_) in enumerate(sorted(filt,key=lambda x:x[0])): av[a]=i+1
ncl=len(av); print(f"  Train:{len(train_s)}, Test:{len(test_s)}, Classes:{ncl}")

from open_clip import create_model_and_transforms, get_tokenizer
CN="hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224"
clip_model,_,ppv=create_model_and_transforms(CN); tok=get_tokenizer(CN); clip_model=clip_model.to(device)

class VDS(Dataset):
    def __init__(s,sa,vo,pp,tk): s.sa=sa; s.vo=vo; s.pp=pp; s.tk=tk; s.u=vo.get('<unk>',0)
    def __len__(s): return len(s.sa)
    def __getitem__(s,i): x=s.sa[i]; return s.pp(x['image']),s.tk([x['question']])[0],s.vo.get(x['answer'],s.u)
def coll(b):
    im,tx,lb=zip(*b); im=torch.stack(im); mx=max(t.shape[0] for t in tx); p=torch.zeros(len(tx),mx,dtype=tx[0].dtype)
    for i,t in enumerate(tx): p[i,:t.shape[0]]=t
    return im,p,torch.tensor(lb,dtype=torch.long)

idx=np.random.permutation(len(train_s)); sz=len(train_s)//NC
cspl={c:idx[c*sz:(c+1)*sz if c<NC-1 else len(train_s)].tolist() for c in range(NC)}
cl,csz={},{}
for cid,indices in cspl.items(): cl[cid]=DataLoader(VDS([train_s[i] for i in indices],av,ppv,tok),batch_size=BS,shuffle=True,num_workers=2,pin_memory=True,collate_fn=coll); csz[cid]=len(indices); print(f"  C{cid}:{len(indices)}")
tl=DataLoader(VDS(test_s,av,ppv,tok),batch_size=BS,shuffle=False,num_workers=2,pin_memory=True,collate_fn=coll)

# ── MODEL ──
class FTL(nn.Module):
    def __init__(s,d,nh,do=0.1):
        super().__init__(); s.v2t=nn.MultiheadAttention(d,nh,dropout=do,batch_first=True); s.vn1=nn.LayerNorm(d); s.vn2=nn.LayerNorm(d)
        s.vf=nn.Sequential(nn.Linear(d,d*4),nn.GELU(),nn.Dropout(do),nn.Linear(d*4,d),nn.Dropout(do))
        s.t2v=nn.MultiheadAttention(d,nh,dropout=do,batch_first=True); s.tn1=nn.LayerNorm(d); s.tn2=nn.LayerNorm(d)
        s.tf=nn.Sequential(nn.Linear(d,d*4),nn.GELU(),nn.Dropout(do),nn.Linear(d*4,d),nn.Dropout(do))
    def forward(s,v,t,k=None): o,_=s.v2t(v,t,t,key_padding_mask=k); v=s.vn1(v+o); v=s.vn2(v+s.vf(v)); o,_=s.t2v(t,v,v); t=s.tn1(t+o); t=s.tn2(t+s.tf(t)); return v,t

class BM(nn.Module):
    def __init__(s,clip,nc):
        super().__init__(); s.clip=clip
        s.fl=nn.ModuleList([FTL(D,HEADS,DROP) for _ in range(FL)])
        s.pq=nn.Parameter(torch.randn(1,1,D)*0.02); s.pa=nn.MultiheadAttention(D,HEADS,dropout=DROP,batch_first=True); s.pn=nn.LayerNorm(D)
        s.head=nn.Sequential(nn.Linear(D,D),nn.GELU(),nn.Dropout(DROP),nn.Linear(D,D//2),nn.GELU(),nn.Dropout(DROP),nn.Linear(D//2,nc))
    def ei(s,img): ve=s.clip.visual; x=ve.trunk.patch_embed(img); x=ve.trunk._pos_embed(x); x=ve.trunk.patch_drop(x); x=ve.trunk.norm_pre(x); x=ve.trunk.blocks(x); return ve.trunk.norm(x)
    def et(s,ids): te=s.clip.text; am=(ids!=0).long(); return te.transformer(input_ids=ids,attention_mask=am).last_hidden_state,am
    def forward(s,img,ids):
        v=s.ei(img); t,am=s.et(ids); kpm=(am==0)
        for f in s.fl: v,t=f(v,t,k=kpm)
        c=torch.cat([v,t],1); B=c.shape[0]; pq=s.pq.expand(B,-1,-1); p,_=s.pa(pq,c,c); return s.head(s.pn(pq+p).squeeze(1))

gm=BM(clip_model,ncl).to(device)
for p in gm.clip.parameters(): p.requires_grad=False
nt=sum(p.numel() for p in gm.parameters()); ntr=sum(p.numel() for p in gm.parameters() if p.requires_grad)
print(f"\n  Total:{nt:,}, Trainable:{ntr:,}")

# ── FedAvg ──
print("\n"+"="*60+"\nTRAINING (FedAvg)\n"+"="*60)
def gp(m): return [p.data.cpu().numpy().copy() for p in m.parameters()]
def sp(m,ps):
    for p,w in zip(m.parameters(),ps): p.data=torch.from_numpy(w).to(p.device)

def agg(cp,sizes):
    total=sum(sizes); wts=[n/total for n in sizes]
    # Wrap the sum in np.array() to ensure 0-D tensors don't become float32 scalars
    return [np.array(sum(wts[i]*cp[i][p] for i in range(len(cp)))) for p in range(len(cp[0]))]

crit=nn.CrossEntropyLoss(label_smoothing=LS)
@torch.no_grad()
def ev(loader):
    gm.eval(); ls,c,t=0.0,0,0
    for i,d,l in loader: i,d,l=i.to(device),d.to(device),l.to(device); lo=gm(i,d); ls+=crit(lo,l).item()*l.size(0); c+=(lo.argmax(-1)==l).sum().item(); t+=l.size(0)
    return ls/t,100*c/t

hist={'round':[],'avg_train_loss':[],'avg_train_acc':[],'test_loss':[],'test_acc':[],'round_time':[]}
for cid in range(NC): hist[f'c{cid}_loss']=[]; hist[f'c{cid}_acc']=[]
ba,bs=0.0,None

for rnd in range(1,ROUNDS+1):
    t0=time.time(); gpar=gp(gm); rcp=[]; rl=[]; rc,rt=0,0
    pb=tqdm(range(NC),desc=f"R{rnd:02d}/{ROUNDS}",leave=False)
    for cid in pb:
        loc=copy.deepcopy(gm); sp(loc,[p.copy() for p in gpar])
        for p in loc.clip.parameters(): p.requires_grad=False
        loc.train(); enc_ids=set(id(p) for p in loc.clip.parameters())
        fhp=[p for p in loc.parameters() if id(p) not in enc_ids and p.requires_grad]
        opt=torch.optim.AdamW(fhp,lr=FLR,weight_decay=WD); cl2,cc,ct=0.0,0,0
        for _ in range(LEP):
            for i,d,l in cl[cid]:
                i,d,l=i.to(device),d.to(device),l.to(device); opt.zero_grad()
                with torch.no_grad(): v=loc.ei(i); t,am=loc.et(d)
                kpm=(am==0)
                for f in loc.fl: v,t=f(v,t,k=kpm)
                c=torch.cat([v,t],1); B=c.shape[0]; pq=loc.pq.expand(B,-1,-1); p,_=loc.pa(pq,c,c)
                lo=loc.head(loc.pn(pq+p).squeeze(1)); loss=crit(lo,l); loss.backward()
                nn.utils.clip_grad_norm_(fhp,1.0); opt.step(); cl2+=loss.item()*l.size(0); cc+=(lo.argmax(-1)==l).sum().item(); ct+=l.size(0)
        rcp.append(gp(loc)); c_l=cl2/max(ct,1); c_a=100*cc/max(ct,1); rl.append(c_l); rc+=cc; rt+=ct
        hist[f'c{cid}_loss'].append(round(c_l,4)); hist[f'c{cid}_acc'].append(round(c_a,2)); del loc,opt
    sp(gm,agg(rcp,list(csz.values()))); tel,tea=ev(tl); rtime=time.time()-t0
    al=np.mean(rl); aa=100*rc/max(rt,1)
    hist['round'].append(rnd); hist['avg_train_loss'].append(round(al,4)); hist['avg_train_acc'].append(round(aa,2))
    hist['test_loss'].append(round(tel,4)); hist['test_acc'].append(round(tea,2)); hist['round_time'].append(round(rtime,1))
    mk=""
    if tea>ba: ba=tea; bs=copy.deepcopy(gm.state_dict()); mk=" ★"
    print(f"R{rnd:02d} [{rtime:.1f}s]  Train:{al:.4f}/{aa:.1f}%  Test:{tel:.4f}/{tea:.1f}%{mk}")
if bs: gm.load_state_dict(bs)
tel,tea=ev(tl); print(f"\n{'='*60}\nFINAL: {tea:.2f}%\n{'='*60}")

import openpyxl; from openpyxl.styles import Font,PatternFill,Alignment
wb=openpyxl.Workbook(); ws=wb.active; ws.title="Training"
hf=Font(name='Arial',bold=True,size=11,color='FFFFFF'); hfi=PatternFill(start_color='1A5276',end_color='1A5276',fill_type='solid')
headers=['Round','Avg Train Loss','Avg Train Acc (%)','Test Loss','Test Acc (%)','Time (s)']
for cid in range(NC): headers+=[f'C{cid} Loss',f'C{cid} Acc (%)']
for c,h in enumerate(headers,1): cl2=ws.cell(row=1,column=c,value=h); cl2.font=hf; cl2.fill=hfi; cl2.alignment=Alignment(horizontal='center')
for i,rnd in enumerate(hist['round']):
    r=i+2; ws.cell(row=r,column=1,value=rnd); ws.cell(row=r,column=2,value=hist['avg_train_loss'][i]); ws.cell(row=r,column=3,value=hist['avg_train_acc'][i])
    ws.cell(row=r,column=4,value=hist['test_loss'][i]); ws.cell(row=r,column=5,value=hist['test_acc'][i]); ws.cell(row=r,column=6,value=hist['round_time'][i])
    for cid in range(NC): ws.cell(row=r,column=7+cid*2,value=hist[f'c{cid}_loss'][i]); ws.cell(row=r,column=8+cid*2,value=hist[f'c{cid}_acc'][i])
ws2=wb.create_sheet("Summary")
for i,(k,v) in enumerate([("Method","FedAvg BiomedCLIP"),("Dataset","VizWiz"),("Total",f"{nt:,}"),("Trainable",f"{ntr:,}"),("Best Test",round(ba,2)),("Final Test",round(tea,2))],1):
    ws2.cell(row=i,column=1,value=k).font=Font(bold=True,name='Arial'); ws2.cell(row=i,column=2,value=v)
for s in [ws,ws2]:
    for col in s.columns: s.column_dimensions[col[0].column_letter].width=max(len(str(c.value or '')) for c in col)+2
p=f"{OUTPUT_DIR}/vizwiz_fedavg_biomedclip_results.xlsx"; wb.save(p); print(f"\nSaved → {p}\nDONE!")